# Mímir sur TPU Colab

Choisir un environnement Colab **TPU-v5e1 ou TPU-v6e1**, puis placer ce checkout dans `/content/tensor-2` (ou adapter `REPO`).
Le modèle, l'optimiseur, l'entraînement et les checkpoints restent dans Mímir C++.
Les opérations des couches sont compilées par un worker JAX/XLA persistant sur un TPU via PJRT/libtpu.
Le runtime CPU n'est pas modifié. Un TPU explicitement demandé mais indisponible provoque une erreur.

Les 72 types de couches calculables sont portés en FP32 ; les callbacks `Lambda` restent hors du runtime, comme sur CPU.
Chaque opération transfère actuellement ses tenseurs entre Mímir et le worker : ce backend de référence vise la correspondance des opérations, sans promesse de performance ni parallélisme multi-TPU.
Installation JAX : https://docs.jax.dev/en/latest/installation.html#google-cloud-tpu


In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO = Path('/content/tensor-2')
assert (REPO / 'CMakeLists.txt').is_file(), 'Placer le checkout Mímir ici ou adapter REPO'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'jax[tpu]'], check=True)
# Sonde dans un processus séparé : le notebook ne conserve pas la propriété du TPU.
subprocess.run([sys.executable, '-c', "import jax; print(jax.devices('tpu'))"], check=True)


In [ ]:
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'cmake', 'ninja-build', 'g++', 'liblua5.3-dev', 'liblz4-dev'], check=True)
subprocess.run(['cmake', '-S', str(REPO), '-B', str(REPO/'build-tpu'), '-G', 'Ninja',
    '-DCMAKE_BUILD_TYPE=Release', '-DMIMIR_ENABLE_TESTS=ON', '-DMIMIR_VIZ_BACKEND=NONE',
    '-DENABLE_SFML=OFF', '-DENABLE_VULKAN=OFF', '-DENABLE_OPENCL=OFF', '-DENABLE_CUDA=OFF',
    '-DENABLE_ROCM=OFF', '-DENABLE_FFMPEG=OFF', '-DENABLE_SCRIPTING_RUST=OFF',
    '-DENABLE_SCRIPTING_JS=OFF', '-DENABLE_SCRIPTING_CSHARP=OFF'], check=True)
subprocess.run(['cmake', '--build', str(REPO/'build-tpu'), '-j', '2', '--target',
    'mimir', 'RuntimeTest.TpuCpuFixture', 'RuntimeTest.TpuModelTrain'], check=True)


In [ ]:
# auto accepte exclusivement TPU-v5e1 et TPU-v6e1 avec un seul périphérique visible.
# Choisir explicitement 'TPU-v5e1' ou 'TPU-v6e1' pour vérifier la cible.
TPU_TARGET = 'auto'
env = dict(os.environ, MIMIR_TPU='1', MIMIR_TPU_TARGET=TPU_TARGET,
           MIMIR_TPU_PYTHON=sys.executable, OMP_NUM_THREADS='1')
# Test matériel : sorties, gradients, accumulation et selected_runtime=TPU.
subprocess.run([str(REPO/'build-tpu/Tests/RuntimeTest.TpuCpuFixture'), '--tpu'],
               cwd=REPO, env=env, check=True)
# Boucle native Mímir : perte décroissante, paramètres mis à jour sur la route TPU.
subprocess.run([str(REPO/'build-tpu/Tests/RuntimeTest.TpuModelTrain')],
               cwd=REPO, env=env, check=True)


Pour utiliser un script Mímir existant, conserver `env` et lancer :

```python
subprocess.run([str(REPO/'build-tpu/bin/mimir'), '--lua', 'scripts/training/ton_script.lua'],
               cwd=REPO, env=env, check=True)
```

Le journal doit afficher `selected_runtime=TPU`. `MIMIR_TPU_DEVICE=0` sélectionne le périphérique ; un seul périphérique est utilisé par processus. `MIMIR_DISABLE_TPU=1` désactive ce backend.
Le test local `Tests/test_tpu_reference.py` utilise XLA sur CPU : il vérifie les formules mais ne valide pas le matériel TPU.
